In [61]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, normalize

from kairos import features
from kairos.data import load_followers, load_labels, load_posts
from kairos.label import author_level
from kairos.metrics import score

SEED = 42
DATA_DIR = Path("data")

In [62]:
labels = load_labels(DATA_DIR)
levelled = author_level(load_posts(DATA_DIR))
post_features = features.build(levelled, load_followers(DATA_DIR))
df = labels.merge(post_features, on=["author", "post"], how="left", validate="one_to_one")
df["text"] = df["text"].fillna("")

numeric_cols = features.CONTENT_NUM + features.HISTORY_NUM + features.ARTEFACT_NUM
categorical_cols = features.CONTENT_CAT
full_train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()

posts: 100%|██████████| 5433/5433 [00:03<00:00, 1635.75file/s]


In [63]:
def make_history_features(levelled_posts):
    t = levelled_posts.sort_values(["author", "posted_at"], kind="mergesort").copy()
    group = t.groupby("author", sort=False)
    previous_reactions = group["log_reactions"].shift(1)
    history = previous_reactions.groupby(t["author"], sort=False)

    t["prev_log_reactions"] = previous_reactions
    for window in (5, 10, 20):
        t[f"past{window}_mean"] = history.transform(
            lambda s: s.rolling(window, min_periods=2).mean()
        )
        t[f"past{window}_std"] = history.transform(
            lambda s: s.rolling(window, min_periods=2).std()
        )
    t["recent5_vs_level"] = t["past5_mean"] - t["level"]
    t["recent10_vs_level"] = t["past10_mean"] - t["level"]

    txt = t["text"].fillna("")
    t["n_exclamations"] = txt.str.count(r"\!")
    t["n_question_marks"] = txt.str.count(r"\?")
    t["n_digits"] = txt.str.count(r"\d")
    t["n_bullets"] = txt.str.count("•")
    t["n_uppercase"] = txt.str.count(r"[A-Z]")
    t["uppercase_ratio"] = t["n_uppercase"] / (txt.str.len() + 1)
    t["avg_word_length"] = txt.str.replace(r"\s+", "", regex=True).str.len() / (
        txt.str.split().str.len().fillna(0) + 1
    )

    cols = ["prev_log_reactions", "recent5_vs_level", "recent10_vs_level",
            "n_exclamations", "n_question_marks", "n_digits", "n_bullets",
            "n_uppercase", "uppercase_ratio", "avg_word_length"]
    for window in (5, 10, 20):
        cols += [f"past{window}_mean", f"past{window}_std"]
    return t[["author", "post"] + cols].fillna(0), cols

def create_author_relative_features(all_post_features):
    t = all_post_features.sort_values(["author", "posted_at"], kind="mergesort").copy()
    by_author = t.groupby("author", sort=False)
    by_type = t.groupby(["author", "content_type"], sort=False)

    t["past_text_length_med"] = by_author["n_chars"].transform(
        lambda s: s.shift(1).expanding(min_periods=5).median()
    )
    t["relative_text_length"] = np.log1p(t["n_chars"]) - np.log1p(t["past_text_length_med"])
    t["past_hashtag_mean"] = by_author["n_hashtags"].transform(
        lambda s: s.shift(1).expanding(min_periods=5).mean()
    )
    t["relative_hashtags"] = t["n_hashtags"] - t["past_hashtag_mean"]

    t["last5_log_reactions_med"] = by_author["log_reactions"].transform(
        lambda s: s.shift(1).rolling(5, min_periods=3).median()
    )
    t["last10_log_reactions_max"] = by_author["log_reactions"].transform(
        lambda s: s.shift(1).rolling(10, min_periods=3).max()
    )
    t["last10_log_reactions_min"] = by_author["log_reactions"].transform(
        lambda s: s.shift(1).rolling(10, min_periods=3).min()
    )
    t["recent_median_minus_level"] = t["last5_log_reactions_med"] - t["level"]

    t["earlier_same_type_count"] = by_type.cumcount().astype(np.float32)
    t["earlier_same_type_mean_resid"] = by_type["resid"].transform(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )
    t["earlier_same_type_std_resid"] = by_type["resid"].transform(
        lambda s: s.shift(1).expanding(min_periods=2).std()
    )


    t["past10_avg_resid"] = by_author["resid"].transform(
        lambda s: s.shift(1).rolling(10, min_periods=3).mean()
    )
    t["past10_positive_share"] = by_author["resid"].transform(
        lambda s: (s.shift(1) > 1.157453).rolling(10, min_periods=3).mean()
    )

    text = t["text"].fillna("")
    t["n_emojis"] = text.str.count(r"[\U0001F300-\U0001FAFF]")
    t["n_colons"] = text.str.count(":")
    t["n_semicolons"] = text.str.count(";")
    t["n_asterisks"] = text.str.count(r"\*")
    t["n_parentheses"] = text.str.count(r"[()]")
    t["n_long_words"] = text.str.count(r"\b\w{10,}\b")
    t["has_bulleted_list"] = text.str.contains(r"[•▪\-]\s", regex=True).astype(np.int8)
    t["text_digit_ratio"] = text.str.count(r"\d") / (text.str.len() + 1)
    t["text_newline_ratio"] = text.str.count("\n") / (text.str.len() + 1)

    new_cols = [
        "past_text_length_med", "relative_text_length", "past_hashtag_mean",
        "relative_hashtags", "last5_log_reactions_med", "last10_log_reactions_max",
        "last10_log_reactions_min", "recent_median_minus_level",
        "earlier_same_type_count", "earlier_same_type_mean_resid",
        "earlier_same_type_std_resid", "past10_avg_resid", "past10_positive_share",
        "n_emojis", "n_colons", "n_semicolons", "n_asterisks",
        "n_parentheses", "n_long_words", "has_bulleted_list",
        "text_digit_ratio", "text_newline_ratio",
    ]
    return t[["author", "post"] + new_cols].fillna(0), new_cols

extra_history, extra_cols = make_history_features(levelled)
relative_all, relative_cols = create_author_relative_features(post_features)

In [64]:
SEMANTIC_DIM = 24
MAX_PREVIOUS = 40

def make_semantic_features(fitting_posts, all_posts):
    corpus = all_posts.sort_values(["author", "posted_at"], kind="mergesort").reset_index(drop=True)
    vectorizer = TfidfVectorizer(
        min_df=3, max_df=0.97, max_features=30000,
        ngram_range=(1, 2), sublinear_tf=True, dtype=np.float32,
    )
    train_matrix = vectorizer.fit_transform(fitting_posts["text"].fillna(""))
    dimension = min(SEMANTIC_DIM, train_matrix.shape[1] - 1)
    svd = TruncatedSVD(n_components=dimension, n_iter=5, random_state=SEED)
    svd.fit(train_matrix)


    embed = svd.transform(vectorizer.transform(corpus["text"].fillna(""))).astype(np.float32)
    embed = normalize(embed, norm="l2", axis=1)
    col_names = [f"svd_{k}" for k in range(dimension)]
    sem = pd.DataFrame(embed, columns=col_names)
    sem.insert(0, "post", corpus["post"].to_numpy())
    sem.insert(0, "author", corpus["author"].to_numpy())

    cos_max = np.zeros(len(corpus), dtype=np.float32)
    cos_mean = np.zeros(len(corpus), dtype=np.float32)
    similar_prior_resid = np.zeros(len(corpus), dtype=np.float32)
    prior_residuals = corpus["resid"].to_numpy(dtype=np.float32)


    for _, indices in corpus.groupby("author", sort=False).indices.items():
        ids = np.asarray(indices, dtype=np.int64)
        for pos in range(1, len(ids)):
            current = ids[pos]
            previous = ids[max(0, pos - MAX_PREVIOUS):pos]
            similarities = embed[previous] @ embed[current]
            cos_max[current] = float(similarities.max())
            cos_mean[current] = float(similarities.mean())
            observed = np.isfinite(prior_residuals[previous])
            if observed.any():
                available = previous[observed]
                similarity_valid = similarities[observed]
                neighbors = available[np.argsort(similarity_valid)[-min(3, len(available)):]]
                similar_prior_resid[current] = float(prior_residuals[neighbors].mean())

    sem["past_text_similarity_max"] = cos_max
    sem["past_text_similarity_mean"] = cos_mean
    sem["similar_past_residual"] = similar_prior_resid
    col_names += ["past_text_similarity_max", "past_text_similarity_mean", "similar_past_residual"]
    return sem, col_names

def make_catboost_tables(fitting_posts, evaluation_posts):
    sem, sem_cols = make_semantic_features(fitting_posts, levelled)

    extra = extra_history.merge(sem, on=["author", "post"], how="inner", validate="one_to_one")
    columns = numeric_cols + extra_cols + sem_cols + categorical_cols
    def attach(source):
        out = source.merge(extra, on=["author", "post"], how="left", validate="one_to_one")
        out[categorical_cols] = out[categorical_cols].fillna("unknown").astype(str)
        out[numeric_cols + extra_cols + sem_cols] = (
            out[numeric_cols + extra_cols + sem_cols].fillna(0).astype(np.float32)
        )
        return out[columns]
    return attach(fitting_posts), attach(evaluation_posts), columns

def extend_cat_table(existing_features, labelled_posts):

    plus = labelled_posts[["author", "post"]].merge(
        relative_all, on=["author", "post"], how="left", validate="one_to_one"
    )
    return pd.concat([
        existing_features.reset_index(drop=True),
        plus[relative_cols].astype(np.float32).reset_index(drop=True)
    ], axis=1)

In [65]:
word_preprocessor = ColumnTransformer([
    ("word", TfidfVectorizer(
        ngram_range=(1, 2), min_df=3, max_features=50000,
        sublinear_tf=True, dtype=np.float32
    ), "text"),
    ("numeric", StandardScaler(), numeric_cols),
    ("categorical", OneHotEncoder(handle_unknown="ignore", min_frequency=20), categorical_cols),
], sparse_threshold=1.0)

word_pipeline = Pipeline([
    ("preprocessor", word_preprocessor),
    ("model", SGDClassifier(
        loss="log_loss", alpha=1e-5, max_iter=1500, tol=1e-3,
        class_weight="balanced", random_state=SEED, average=True
    )),
])

def make_catboost(y):
    counts = np.bincount(y.to_numpy(), minlength=3)
    weights = (len(y) / (3 * counts)).tolist()
    return CatBoostClassifier(
        depth=7, iterations=850, learning_rate=0.04, l2_leaf_reg=8,
        loss_function="MultiClass", class_weights=weights,
        random_seed=SEED, cat_features=categorical_cols,
        thread_count=4, verbose=False, allow_writing_files=False
    )

def predict_ensemble(word_model, cat_model, word_data, cat_data):
    probabilities = (
        0.15 * word_model.predict_proba(word_data)
        + 0.85 * cat_model.predict_proba(cat_data)
    )
    return np.argmax(probabilities * [0.8, 1.0, 0.7], axis=1)

In [66]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_indices, val_indices = next(splitter.split(full_train, groups=full_train["author"]))
train = full_train.iloc[train_indices].copy()
val = full_train.iloc[val_indices].copy()

word_model = clone(word_pipeline).fit(train, train["label"])
cat_train, cat_val, _ = make_catboost_tables(train, val)
cat_train = extend_cat_table(cat_train, train)
cat_val = extend_cat_table(cat_val, val)
cat_model = make_catboost(train["label"])
cat_model.fit(cat_train, train["label"])
val_prediction = predict_ensemble(word_model, cat_model, val, cat_val)
print("Validation:", score(val["label"], val_prediction))

Validation: {'macro_f1': 0.6498, 'qwk': 0.5856, 'accuracy': 0.6796}


In [67]:
word_model = clone(word_pipeline).fit(full_train, full_train["label"])
cat_train, cat_test, _ = make_catboost_tables(full_train, test)
cat_train = extend_cat_table(cat_train, full_train)
cat_test = extend_cat_table(cat_test, test)
cat_model = make_catboost(full_train["label"])
cat_model.fit(cat_train, full_train["label"])

prediction = predict_ensemble(word_model, cat_model, test, cat_test)
print("Test:", score(test["label"], prediction))
print("Confusion matrix [0, 1, 2]:")
print(confusion_matrix(test["label"], prediction, labels=[0, 1, 2]))

test[["author", "post"]].assign(label=prediction).to_csv("lab1_predictions.csv", index=False)

Test: {'macro_f1': 0.6355, 'qwk': 0.5705, 'accuracy': 0.6649}
Confusion matrix [0, 1, 2]:
[[2063 1430   48]
 [2017 7329 1145]
 [ 109 1156 2326]]
